# Participant and condition diagnostics

**Purpose:** Summarise held-out prediction errors by participant, speed and relative cadence.

**Required inputs:** 04 and 05 results ZIPs.

**Outputs:** Trial, participant and condition diagnostic tables.

**Usage:** Run after 04 and 05. Files are read from the notebook’s current working directory unless specified in its configuration.


## Environment and input archives


In [ ]:
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

BASE = Path.cwd()
ZIP04 = BASE / "04_Bacek_Foundational_Predictive_Models_results.zip"
ZIP05 = BASE / "05_Bacek_Incremental_Gait_Information_results.zip"
OUTPUT_NAME = "13_Bacek_Participant_Condition_Error_Diagnostics_results"
OUTPUT_DIR = BASE / OUTPUT_NAME
OUTPUT_ZIP = BASE / f"{OUTPUT_NAME}.zip"

for path in (ZIP04, ZIP05):
    if not path.is_file():
        raise FileNotFoundError(f"Required results archive not found: {path.name}")

SOURCE_SHA256 = "c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3"
FOLDS = {
    1: {"S08", "S10", "S16", "S21"},
    2: {"S11", "S13", "S17", "S20"},
    3: {"S01", "S02", "S05", "S18"},
    4: {"S03", "S07", "S12", "S14", "S19"},
    5: {"S04", "S06", "S09", "S15"},
}
EXPECTED_COUNTS = {1: 55, 2: 53, 3: 56, 4: 65, 5: 55}
SPEEDS = [0.4, 0.8, 1.1]
CADENCES = [0.90, 0.95, 1.00, 1.10, 1.20]
PROCEDURES = ["B0", "G", "C", "C+G"]

checks = []
def check(name, condition):
    if not bool(condition):
        raise AssertionError(f"Verification failed: {name}")
    checks.append({"check": name, "status": "PASS"})

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_csv(z, filename):
    return pd.read_csv(BytesIO(z.read(filename)))

started_at = datetime.now(timezone.utc).isoformat()
print("Input archives:", ZIP04.name, "and", ZIP05.name)
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)

## Verify the source identities, 284-trial ledger and reused procedures


In [ ]:
sha04 = sha256_file(ZIP04)
sha05 = sha256_file(ZIP05)
with zipfile.ZipFile(ZIP04) as z04, zipfile.ZipFile(ZIP05) as z05:
    summary04 = json.loads(z04.read("run_summary.json"))
    summary05 = json.loads(z05.read("run_summary.json"))
    verify04 = read_csv(z04, "verification_checks.csv")
    verify05 = read_csv(z05, "verification_checks.csv")
    ledger04 = read_csv(z04, "eligible_trial_ledger.csv")
    ledger05 = read_csv(z05, "eligible_trial_ledger.csv")
    predictions04 = read_csv(z04, "held_out_predictions.csv")
    predictions05 = read_csv(z05, "held_out_predictions.csv")
    metrics04 = read_csv(z04, "pooled_metrics.csv")
    metrics05 = read_csv(z05, "pooled_metrics.csv")

check("prior_archives_report_pass", summary04["status"] == summary05["status"] == "PASS")
check("prior_archives_verification_pass", verify04["status"].eq("PASS").all() and verify05["status"].eq("PASS").all())
check("authoritative_source_fingerprints", summary04["source_sha256"] == summary05["source_sha256"] == SOURCE_SHA256)
check("notebook05_references_exact_notebook04_archive", summary05["reference_notebook04_zip_sha256"] == sha04)

key = "trial_uid"
common_ledger = ["trial_uid", "subject_id", "trial_id", "outer_fold", "retained_cycles",
                 "speed_m_s", "relative_cadence", "condition_index", "observed_W_per_kg"]
check("ledger_columns_available", all(c in ledger04.columns and c in ledger05.columns for c in common_ledger))
ledger = ledger04[common_ledger].sort_values(key).reset_index(drop=True)
other_ledger = ledger05[common_ledger].sort_values(key).reset_index(drop=True)
check("source_ledgers_identical", ledger.equals(other_ledger))
check("primary_trial_count_and_unique_ids", len(ledger) == 284 and ledger[key].is_unique)
check("participant_and_cycle_totals", ledger.subject_id.nunique() == 21 and int(ledger.retained_cycles.sum()) == 60757)
check("outer_fold_counts", ledger.outer_fold.value_counts().to_dict() == EXPECTED_COUNTS)
check("outer_fold_participant_memberships", all(set(ledger.loc[ledger.outer_fold == fold, "subject_id"]) == people for fold, people in FOLDS.items()))
check("trial_id_to_uid_identity", (ledger.subject_id + "_" + ledger.trial_id).equals(ledger.trial_uid))

# Independently check the prescribed T01–T15 experimental condition mapping.
def condition_for_trial(trial_id):
    n = int(trial_id.removeprefix("T"))
    if not 1 <= n <= 15:
        raise AssertionError(f"Unexpected unrestricted trial ID: {trial_id}")
    speed = 1.1 if n <= 5 else 0.8 if n <= 10 else 0.4
    cadence = CADENCES[(n - 1) % 5]
    return speed, cadence
expected_conditions = [condition_for_trial(t) for t in ledger.trial_id]
check("verified_speed_and_cadence_mapping", all(
    np.isclose(row.speed_m_s, s, atol=1e-12) and np.isclose(row.relative_cadence, c, atol=1e-12)
    for row, (s, c) in zip(ledger.itertuples(index=False), expected_conditions)
))
check("all_15_condition_cells_present", len(ledger.groupby(["speed_m_s", "relative_cadence"])) == 15)

required_prediction_columns = {"procedure", "outer_fold", "subject_id", "trial_id",
                               "trial_uid", "observed_W_per_kg", "predicted_W_per_kg"}
check("prediction_columns_available", required_prediction_columns.issubset(predictions04.columns) and required_prediction_columns.issubset(predictions05.columns))
check("source_predictions_finite", np.isfinite(predictions04[["observed_W_per_kg", "predicted_W_per_kg"]]).all().all() and np.isfinite(predictions05[["observed_W_per_kg", "predicted_W_per_kg"]]).all().all())

# The Notebook 05 C and G predictions must exactly match Notebook 04 B2 and G_SELECTED.
def sorted_procedure(frame, procedure):
    return frame.loc[frame.procedure == procedure].sort_values(key).reset_index(drop=True)
for name04, name05 in [("B2", "C"), ("G_SELECTED", "G")]:
    a = sorted_procedure(predictions04, name04)
    b = sorted_procedure(predictions05, name05)
    check(f"reference_{name04}_equals_{name05}",
          len(a) == len(b) == 284 and
          a[["trial_uid", "subject_id", "trial_id", "outer_fold"]].equals(
              b[["trial_uid", "subject_id", "trial_id", "outer_fold"]]) and
          np.array_equal(a.observed_W_per_kg.to_numpy(), b.observed_W_per_kg.to_numpy()) and
          np.array_equal(a.predicted_W_per_kg.to_numpy(), b.predicted_W_per_kg.to_numpy()))

selected = []
for frame, source_name, output_name in [
    (predictions04, "B0", "B0"), (predictions04, "G_SELECTED", "G"),
    (predictions05, "C", "C"), (predictions05, "C+G", "C+G")
]:
    part = sorted_procedure(frame, source_name).copy()
    check(f"{output_name}_complete_unique_trial_predictions", len(part) == 284 and part.trial_uid.is_unique and set(part.trial_uid) == set(ledger.trial_uid))
    part["procedure"] = output_name
    selected.append(part)
predictions = pd.concat(selected, ignore_index=True)
diagnostic = predictions.merge(ledger, on="trial_uid", validate="many_to_one", suffixes=("_pred", "_src"))
for c in ["subject_id", "trial_id", "outer_fold"]:
    check(f"prediction_matches_source_{c}", (diagnostic[f"{c}_pred"] == diagnostic[f"{c}_src"]).all())
check("prediction_target_matches_source", np.array_equal(diagnostic.observed_W_per_kg_pred.to_numpy(), diagnostic.observed_W_per_kg_src.to_numpy()))

diagnostic = diagnostic.rename(columns={
    "subject_id_src": "subject_id", "trial_id_src": "trial_id", "outer_fold_src": "outer_fold",
    "observed_W_per_kg_src": "observed_W_per_kg"}).drop(columns=[
    "subject_id_pred", "trial_id_pred", "outer_fold_pred", "observed_W_per_kg_pred"])
diagnostic["residual_W_per_kg"] = diagnostic.predicted_W_per_kg - diagnostic.observed_W_per_kg
diagnostic["absolute_error_W_per_kg"] = diagnostic.residual_W_per_kg.abs()
diagnostic["squared_error_W2_per_kg2"] = diagnostic.residual_W_per_kg**2

diagnostic = diagnostic.sort_values(["procedure", "subject_id", "trial_id"]).reset_index(drop=True)
check("diagnostic_row_count", len(diagnostic) == 4 * 284)
print(f"Verified {len(ledger)} source trials and {len(diagnostic)} matched predictions across four procedures.")

## Independent metric calculations


In [ ]:
def summarise(frame):
    error = frame.residual_W_per_kg.to_numpy(dtype=np.float64)
    return {
        "n_trials": int(len(frame)),
        "n_participants": int(frame.subject_id.nunique()),
        "RMSE_W_per_kg": float(np.sqrt(np.mean(error * error))),
        "MAE_W_per_kg": float(np.mean(np.abs(error))),
        "bias_W_per_kg": float(np.mean(error)),
    }

def group_summary(columns):
    rows = []
    for group_values, part in diagnostic.groupby(columns, sort=True):
        vals = group_values if isinstance(group_values, tuple) else (group_values,)
        rows.append(dict(zip(columns, vals)) | summarise(part))
    return pd.DataFrame(rows).sort_values(columns).reset_index(drop=True)

pooled = group_summary(["procedure"])
participant = group_summary(["procedure", "subject_id", "outer_fold"])
speed = group_summary(["procedure", "speed_m_s"])
cadence = group_summary(["procedure", "relative_cadence"])
condition = group_summary(["procedure", "speed_m_s", "relative_cadence"])

# The prior experiments' pooled metrics are independent saved values.
for proc in PROCEDURES:
    reported = metrics04.loc[metrics04.procedure == ({"G": "G_SELECTED"}.get(proc, proc))] if proc in ("B0", "G") else metrics05.loc[metrics05.procedure == proc]
    computed = pooled.loc[pooled.procedure == proc]
    check(f"{proc}_pooled_metrics_reproduce_prior_result", len(reported) == len(computed) == 1 and
          all(np.isclose(float(reported.iloc[0][k]), float(computed.iloc[0][k]), atol=1e-10, rtol=0)
              for k in ["RMSE_W_per_kg", "MAE_W_per_kg", "bias_W_per_kg"]))

check("all_21_participants_per_procedure", len(participant) == 4 * 21)
check("exact_3_speed_strata_per_procedure", len(speed) == 4 * 3)
check("exact_5_cadence_strata_per_procedure", len(cadence) == 4 * 5)
check("exact_15_condition_cells_per_procedure", len(condition) == 4 * 15)
for name, frame in [("participant", participant), ("speed", speed), ("cadence", cadence), ("condition", condition)]:
    check(f"{name}_trial_counts_reconcile", all(int(frame.loc[frame.procedure == p, "n_trials"].sum()) == 284 for p in PROCEDURES))
    check(f"{name}_participant_counts_valid", frame.n_participants.between(1, 21).all())
check("all_stratum_metrics_finite", all(np.isfinite(frame[["RMSE_W_per_kg", "MAE_W_per_kg", "bias_W_per_kg"]].to_numpy()).all() for frame in [pooled, participant, speed, cadence, condition]))

c_metrics = participant.loc[participant.procedure == "C", ["subject_id", "outer_fold", "n_trials", "RMSE_W_per_kg", "MAE_W_per_kg"]].rename(columns={"RMSE_W_per_kg": "RMSE_C_W_per_kg", "MAE_W_per_kg": "MAE_C_W_per_kg"})
cg_metrics = participant.loc[participant.procedure == "C+G", ["subject_id", "outer_fold", "n_trials", "RMSE_W_per_kg", "MAE_W_per_kg"]].rename(columns={"RMSE_W_per_kg": "RMSE_C_plus_G_W_per_kg", "MAE_W_per_kg": "MAE_C_plus_G_W_per_kg"})
paired = c_metrics.merge(cg_metrics, on=["subject_id", "outer_fold", "n_trials"], validate="one_to_one")
paired["delta_RMSE_C_minus_C_plus_G_W_per_kg"] = paired.RMSE_C_W_per_kg - paired.RMSE_C_plus_G_W_per_kg
paired["delta_MAE_C_minus_C_plus_G_W_per_kg"] = paired.MAE_C_W_per_kg - paired.MAE_C_plus_G_W_per_kg
check("all_21_paired_participants_present", len(paired) == 21 and paired.subject_id.is_unique)

print("Overall descriptive metrics (W/kg):")
print(pooled[["procedure", "n_trials", "RMSE_W_per_kg", "MAE_W_per_kg", "bias_W_per_kg"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nStratum row counts:", len(participant), "participants,", len(speed), "speed,", len(cadence), "cadence,", len(condition), "condition-cell")

## Diagnostic figures (optional for the report)


In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

# Figure A: matched participant-level change in RMSE (C minus C+G).
ordered = paired.sort_values("delta_RMSE_C_minus_C_plus_G_W_per_kg")
fig, ax = plt.subplots(figsize=(8.2, 7.3))
y_pos = np.arange(len(ordered))
ax.barh(y_pos, ordered["delta_RMSE_C_minus_C_plus_G_W_per_kg"])
ax.set_yticks(y_pos, [f"{s}  (n={n})" for s, n in zip(ordered.subject_id, ordered.n_trials)])
ax.axvline(0, color="black", linewidth=0.9)
ax.set_xlabel("RMSE(C) − RMSE(C+G) (W/kg)")
ax.set_ylabel("Participant (held-out trial count)")
ax.set_title("Participant-level effect of adding gait to conditions")
ax.text(0.01, -0.09, "Positive: C+G lower RMSE     Negative: C lower RMSE", transform=ax.transAxes, fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "E5_1_Participant_Paired_RMSE_Differences.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "E5_1_Participant_Paired_RMSE_Differences.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Figures B/C: matching limits in both panels prevent misleading visual comparisons.
subsets = {name: diagnostic.loc[diagnostic.procedure == name] for name in ["C", "C+G"]}
values = np.concatenate([subsets[k]["observed_W_per_kg"].to_numpy() for k in subsets] + [subsets[k]["predicted_W_per_kg"].to_numpy() for k in subsets])
lo, hi = float(values.min()), float(values.max())
pad = 0.05 * (hi - lo)
lo, hi = lo - pad, hi + pad

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), sharex=True, sharey=True)
for ax, name in zip(axes, ["C", "C+G"]):
    data = subsets[name]
    ax.scatter(data.observed_W_per_kg, data.predicted_W_per_kg, s=15, alpha=0.5)
    ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1)
    ax.set(xlim=(lo, hi), ylim=(lo, hi), xlabel="Observed net metabolic power (W/kg)", title=name)
    ax.set_aspect("equal", adjustable="box")
axes[0].set_ylabel("Predicted net metabolic power (W/kg)")
fig.suptitle("Condition-only and condition-plus-gait predictions")
fig.tight_layout()
fig.savefig(FIG_DIR / "E5_1_Predicted_vs_Observed_C_and_C_plus_G.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "E5_1_Predicted_vs_Observed_C_and_C_plus_G.png", dpi=180, bbox_inches="tight")
plt.close(fig)

resid_all = diagnostic.loc[diagnostic.procedure.isin(["C", "C+G"]), "residual_W_per_kg"]
rabs = float(np.abs(resid_all).max()) * 1.06
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.7), sharex=True, sharey=True)
for ax, name in zip(axes, ["C", "C+G"]):
    data = subsets[name]
    ax.scatter(data.observed_W_per_kg, data.residual_W_per_kg, s=15, alpha=0.5)
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.set(xlim=(lo, hi), ylim=(-rabs, rabs), xlabel="Observed net metabolic power (W/kg)", title=name)
axes[0].set_ylabel("Residual: predicted − observed (W/kg)")
fig.suptitle("Residual behaviour across observed metabolic power")
fig.tight_layout()
fig.savefig(FIG_DIR / "E5_1_Residual_vs_Observed_C_and_C_plus_G.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "E5_1_Residual_vs_Observed_C_and_C_plus_G.png", dpi=180, bbox_inches="tight")
plt.close(fig)

check("all_three_diagnostic_figures_created", len(list(FIG_DIR.glob("*.pdf"))) == 3 and len(list(FIG_DIR.glob("*.png"))) == 3)
print("Created three diagnostic figures in PDF and PNG formats.")

## Save evidence and create the results ZIP


In [ ]:
# Save the auditable trial-level ledger and descriptive summaries.
files = {
    "trial_diagnostics.csv": diagnostic,
    "pooled_metrics.csv": pooled,
    "participant_metrics.csv": participant,
    "speed_strata_metrics.csv": speed,
    "cadence_strata_metrics.csv": cadence,
    "speed_cadence_cell_metrics.csv": condition,
    "participant_paired_C_vs_C_plus_G.csv": paired.sort_values("subject_id"),
}
for filename, frame in files.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, float_format="%.17g")

check("all_expected_tables_prepared", len(files) == 7 and all((OUTPUT_DIR / filename).is_file() for filename in files))

# A code fingerprint follows the same concatenated-code-cell basis as earlier notebooks.
notebook_filename = "13_Bacek_Participant_Condition_Error_Diagnostics.ipynb"
notebook_file = BASE / notebook_filename
if notebook_file.is_file():
    notebook_data = json.loads(notebook_file.read_text(encoding="utf-8"))
    code_sources = [c["source"] if isinstance(c["source"], str) else "".join(c["source"])
                    for c in notebook_data["cells"] if c["cell_type"] == "code"]
    notebook_code_sha256 = hashlib.sha256("".join(code_sources).encode("utf-8")).hexdigest()
else:
    notebook_code_sha256 = None

summary = {
    "status": "PASS",
    "experiment": "E5.1",
    "prediction_setting": "new-person; internally held-out and exploratory-informed",
    "target": "stored corrected trial-level metcost_mean, net W/kg",
    "source_sha256": SOURCE_SHA256,
    "notebook04_results_zip": ZIP04.name,
    "notebook04_zip_sha256": sha04,
    "notebook05_results_zip": ZIP05.name,
    "notebook05_zip_sha256": sha05,
    "notebook04_code_sha256": summary04.get("notebook_code_sha256"),
    "notebook05_code_sha256": summary05.get("notebook_code_sha256"),
    "notebook_filename": notebook_filename,
    "notebook_code_sha256": notebook_code_sha256,
    "code_fingerprint_basis": "concatenated code-cell sources, UTF-8",
    "procedures": PROCEDURES,
    "n_trials_per_procedure": 284,
    "n_participants": 21,
    "n_total_prediction_rows": len(diagnostic),
    "stratifications": ["participant", "prescribed speed", "relative cadence", "speed x cadence cell"],
    "signed_bias_definition": "predicted - observed (W/kg)",
    "primary_E5_1_scope": "descriptive subgroup diagnostics only",
    "uncertainty_included": False,
    "participant_influence_included": False,
    "model_refitting_performed": False,
    "requirements_evidence": {
        "IR-7": "participant, speed and cadence diagnostics prepared; cluster confidence intervals pending E5.2",
        "IR-8": "participant influence and selection stability pending E5.2",
        "EC-5": "matched prediction identities and finite scores checked",
        "RC-1": "source, upstream-run and code fingerprints recorded",
        "RC-2": "pooled metrics independently recalculated and matched to saved prior results"
    },
    "pooled_RMSE_W_per_kg": pooled.set_index("procedure")["RMSE_W_per_kg"].to_dict(),
    "software": {
        "python": sys.version.split()[0], "numpy": np.__version__,
        "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
        "platform": platform.platform()
    },
    "started_at_utc": started_at,
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "verification": {"passed": len(checks), "failed": 0},
}

(OUTPUT_DIR / "verification_checks.csv").write_text(pd.DataFrame(checks).to_csv(index=False), encoding="utf-8")
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

# Exactly seven diagnostic tables, six plot files and two provenance files.
output_files = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
assert len(output_files) == 15, f"Unexpected output-file count: {len(output_files)}"
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_files:
        archive.write(path, arcname=path.relative_to(OUTPUT_DIR).as_posix())
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    assert len(archive.namelist()) == 15, "Incomplete results ZIP"
    assert archive.testzip() is None, "Corrupt results ZIP"

print(f"PASS: {len(checks)} verification checks")
print("Results directory:", OUTPUT_DIR.name)
print("Share this archive:", OUTPUT_ZIP.name)
print("Cluster intervals, influence and selection stability are reserved for E5.2.")